###### 22_nlp_embeddings

###### Purpose
The purpose of this notebook is to generate embeddings for support ticket text using an embedding model and produce dense vector representations that capture the semantic meaning of the text. These embeddings will be used in later notebooks for Vector Search, RAG, and Agentic AI applications.

###### Technologies Used

- Databricks
- Delta Lake
- Unity Catalog

###### Input

- Delta Table: dbw_agentic_ai_dev.support_ticket_ai.gold_support_ticket_features
- Embedding Model
    - Model Name: databricks-gte-large-en
    - Purpose: Convert support ticket text into dense numerical vectors that capture semantic meaning.


######  Output

Delta Table: dbw_agentic_ai_dev.support_ticket_ai.gold_support_ticket_embeddings



######  Architecture

```text

Gold Support Ticket Features
       ↓
Support Ticket Text
       ↓
Embedding Model
(databricks-gte-large-en)
        ↓
Dense Vector Embeddings
        ↓
Support Ticket Embeddings Delta Table

```


What is an embedding? --> An embedding is a dense numerical vector that represents the semantic meaning of text.
Why do we need embeddings? --> Embedding models convert text into vectors because Vector Search compares vectors, not raw text.
Why are vectors better than TF-IDF? --> TF-IDF matches important words, while embeddings capture the meaning of the entire sentence, allowing semantic search.
Why do similar sentences have similar vectors? --> Embedding models place semantically similar sentences close together in vector space.
What is vector dimension? ---> Vector dimension is the number of numerical values contained in an embedding vector.
What is cosine similarity? ---> Cosine similarity measures how similar two vectors are by comparing the angle between them rather than their length.
Why do we store embeddings? ---> Embeddings are stored in Delta tables so they can later be indexed by Vector Search and used for semantic retrieval, RAG, and Agentic AI applications.

###### Generate Embeddings for the Support Ticket Text 

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage

w = WorkspaceClient()

# Read support ticket data
cleaned_ticket_text = (spark.table("dbw_agentic_ai_dev.support_ticket_ai.gold_support_ticket_features").toPandas())

# Generate Embeddings
embedding_rows = []

for row in cleaned_ticket_text.itertuples():
    response = w.serving_endpoints.query(name="databricks-gte-large-en", input=[row.cleaned_ticket_text])
    embedding = response.data[0].embedding
    embedding_rows.append((row.ticket_id, row.cleaned_ticket_text, row.category, [float(x) for x in embedding]))

# Create Schema
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, ArrayType, DoubleType

schema = StructType([
    StructField("ticket_id", StringType(), True),
    StructField("cleaned_ticket_text", StringType(), True),
    StructField("category", StringType(), True),
    StructField("embedding", ArrayType(DoubleType()), True)
])

# Create dataframe
embedding_df = spark.createDataFrame(
    embedding_rows,
    schema=schema
)

#display(embedding_df)

#Save Embeddings to the delta table
embedding_df.write.format("delta").mode("overwrite").saveAsTable("dbw_agentic_ai_dev.support_ticket_ai.gold_support_ticket_embeddings")

#Verify
display(spark.table("dbw_agentic_ai_dev.support_ticket_ai.gold_support_ticket_embeddings"))


###### Notebook Summary
- Load support ticket data from delta gold table
- Generate dense vector embeddings for each support ticket using the Databricks GTE embedding model.
- Create Embedding Table with Schema
- Save Embeddings to the delta table - dbw_agentic_ai_dev.support_ticket_ai.gold_support_ticket_embeddings


######  Key Learnings

- Embedding models convert text into dense numerical vectors.
- Similar meanings produce similar vectors.
- Embeddings capture semantics rather than exact word matches.
- Embeddings are stored in Delta because Vector Search indexes vectors, not raw text.
``` text

Traditional NLP
↓

TF-IDF
↓

Machine Learning
```

-------------------------

``` text

Modern NLP

↓

Embedding Model

↓

Vector

↓

Vector Search

↓

LLM

```

- Embeddings replace TF-IDF as the primary text representation in modern AI systems.
- Raw text cannot be compared mathematically. Embeddings convert text into vectors so similarity calculations like cosine similarity can be performed efficiently.

###### Concepts Learned

- Embedding
- Dense Vector
- Semantic Meaning
- Vector Dimension
- Cosine Similarity
- Embedding Model
- Vector Representation


###### Next Notebook

23_nlp_vector_search

The Purpose of this notebook is to create veactor search endpoint and Vector Search Index for Support ticket data. This will be used by the Semantic Search and RAG. 